# Seguridad y límites en Computer Use

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/InteligenciaArtificial/blob/main/tutoriales/notebooks/computer-use-operator/04-seguridad-y-limites.ipynb)

Guardrails, human-in-the-loop y auditoría para despliegues seguros.

In [ ]:
!pip install anthropic -q

## 1. Validación de instrucciones entrantes

In [ ]:
ACCIONES_PROHIBIDAS = [
    'transferencia bancaria', 'pago', 'eliminar base de datos',
    'borrar todos', 'formatear disco', 'enviar a todos',
    'cambiar contraseña de administrador',
]

def validar_instruccion(instruccion: str) -> tuple[bool, str]:
    instruccion_lower = instruccion.lower()
    for accion in ACCIONES_PROHIBIDAS:
        if accion in instruccion_lower:
            return False, f'Acción prohibida detectada: "{accion}"'
    return True, 'OK'

# Tests
instrucciones = [
    ('Abre el navegador y ve a google.com', True),
    ('Realiza una transferencia bancaria de 5000€', False),
    ('Introduce la factura en el ERP y guárdala', True),
    ('Eliminar base de datos de producción', False),
    ('Genera un informe PDF del mes de abril', True),
]

print('Validación de instrucciones:')
for instruccion, esperado in instrucciones:
    permitida, motivo = validar_instruccion(instruccion)
    icono = '✅' if permitida == esperado else '❌'
    estado = '✅ PERMITIDA' if permitida else f'🚫 BLOQUEADA ({motivo})'
    print(f'{icono} {estado}: {instruccion[:60]}')

## 2. Lista blanca de URLs

In [ ]:
from urllib.parse import urlparse

URLS_PERMITIDAS = {'erp.empresa.com', 'portal.proveedor.com', 'docs.empresa.com'}

def guardrail_url(url: str) -> bool:
    dominio = urlparse(url).netloc.lower().removeprefix('www.')
    if dominio not in URLS_PERMITIDAS:
        raise PermissionError(f'URL no permitida: {dominio}. Permitidas: {URLS_PERMITIDAS}')
    return True

urls_test = [
    'https://erp.empresa.com/facturas/nueva',
    'https://portal.proveedor.com/pedidos',
    'https://banco.com/transferencia',
    'https://redes-sociales.com/publicar',
]

for url in urls_test:
    try:
        guardrail_url(url)
        print(f'✅ Permitida: {url}')
    except PermissionError as e:
        print(f'🚫 Bloqueada: {url}')

## 3. Human-in-the-loop para acciones críticas

In [ ]:
ACCIONES_CRITICAS = ['guardar y contabilizar', 'enviar email', 'confirmar pedido', 'publicar']

def requiere_aprobacion(accion: str) -> bool:
    return any(crit in accion.lower() for crit in ACCIONES_CRITICAS)

def simular_hitl(acciones: list[str], auto_aprobar: bool = False) -> list[dict]:
    """Simula el flujo human-in-the-loop."""
    resultados = []
    for accion in acciones:
        if requiere_aprobacion(accion):
            if auto_aprobar:
                aprobada = True
                print(f'⚠️  REQUIERE APROBACIÓN (auto): {accion}')
            else:
                # En producción: esperar input del usuario
                aprobada = True  # Simulamos aprobación
                print(f'⚠️  PAUSADO para aprobación humana: {accion}')
        else:
            aprobada = True
            print(f'🤖 Ejecutando automáticamente: {accion}')
        resultados.append({'accion': accion, 'aprobada': aprobada, 'requeria_aprobacion': requiere_aprobacion(accion)})
    return resultados

acciones_workflow = [
    'Navegar a ERP',
    'Rellenar campos de factura',
    'Verificar importes',
    'Guardar y contabilizar',  # Esta requiere aprobación
    'Imprimir comprobante',
]

resultados = simular_hitl(acciones_workflow)
print(f'\nTotal: {len(resultados)} acciones, {sum(1 for r in resultados if r["requeria_aprobacion"])} requirieron aprobación')

## 4. Auditoría de acciones

In [ ]:
import json
from datetime import datetime

log_acciones = []

def registrar_accion(herramienta: str, entrada: dict, resultado: str, usuario: str = 'sistema'):
    entrada_segura = {k: '***' if k in ('password', 'token', 'secret') else v for k, v in entrada.items()}
    registro = {
        'timestamp': datetime.now().isoformat(),
        'usuario': usuario,
        'herramienta': herramienta,
        'entrada': entrada_segura,
        'resultado_preview': resultado[:100],
    }
    log_acciones.append(registro)
    return registro

# Simular sesión con auditoría
registrar_accion('computer', {'action': 'screenshot'}, 'imagen capturada')
registrar_accion('computer', {'action': 'left_click', 'coordinate': [450, 300]}, 'click OK')
registrar_accion('bash', {'command': 'ls /tmp/output/', 'password': 'secreto123'}, 'archivo1.pdf\narchivo2.pdf')

print('Log de auditoría (credenciales redactadas):')
for r in log_acciones:
    print(f"  [{r['timestamp'][:19]}] {r['herramienta']} → {r['entrada']}")

## Checklist de seguridad

```
✅ Entorno Docker aislado con red restringida
✅ Lista blanca de URLs configurada
✅ Validación de instrucciones entrantes
✅ Human-in-the-loop para acciones irreversibles
✅ Logging completo con redacción de credenciales
✅ Límite de iteraciones configurado (max 30-50)
✅ Credenciales en variables de entorno (nunca hardcodeadas)
```